In [ ]:

print("Football Player Tracking and Analysis")
print("Project started successfully!")


In [ ]:

import torch

if torch.cuda.is_available():
    print("GPU is available!")
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("GPU is not available. We can continue using CPU.")


In [ ]:

!pip install ultralytics opencv-python pandas matplotlib lap


In [ ]:

import cv2
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

print("OpenCV version:", cv2.__version__)
print("Pandas version:", pd.__version__)
print("GPU available:", torch.cuda.is_available())
print("All libraries imported successfully!")


In [ ]:

from google.colab import files

uploaded = files.upload()


In [ ]:

import os

print("Uploaded files:")

for filename in uploaded.keys():
    print(filename, "-", round(os.path.getsize(filename) / (1024 * 1024), 2), "MB")


In [ ]:

import os
import cv2

video_path = "/content/istockphoto-2245307722-640_adpp_is (1).mp4"

if os.path.exists(video_path):
    print("Video uploaded successfully!")

    cap = cv2.VideoCapture(video_path)

    if cap.isOpened():
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        print("Total frames:", total_frames)
        print("FPS:", fps)
        print("Video width:", width)
        print("Video height:", height)

        ret, frame = cap.read()
        print("First frame readable:", ret)

        cap.release()
    else:
        print("Could not open the video.")
else:
    print("Video not found. Please check the filename.")


In [ ]:

import matplotlib.pyplot as plt

cap = cv2.VideoCapture(video_path)
ret, frame = cap.read()
cap.release()

if ret:
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12, 6))
    plt.imshow(frame_rgb)
    plt.title("Football Video - First Frame")
    plt.axis("off")
    plt.show()
else:
    print("Unable to display the video frame.")


In [ ]:

from ultralytics import YOLO

# Load the YOLO11 nano model
model = YOLO("yolo11n.pt")

print("YOLO11 model loaded successfully!")


In [ ]:

import cv2
import matplotlib.pyplot as plt

# Read the first frame
cap = cv2.VideoCapture(video_path)
ret, frame = cap.read()
cap.release()

if ret:
    # Detect objects in the frame
    results = model.predict(frame, conf=0.25, verbose=False)

    # Draw bounding boxes and labels
    annotated_frame = results[0].plot()

    # Convert BGR to RGB for displaying
    annotated_frame = cv2.cvtColor(
        annotated_frame, cv2.COLOR_BGR2RGB
    )

    plt.figure(figsize=(14, 7))
    plt.imshow(annotated_frame)
    plt.title("YOLO11 - Football Player Detection")
    plt.axis("off")
    plt.show()

    print("Detection completed successfully!")
    print("Objects detected:", len(results[0].boxes))
else:
    print("Could not read the video frame.")


In [ ]:

import cv2
from ultralytics import YOLO
from IPython.display import display, Video

# Input and output video paths
video_path = "/content/istockphoto-2245307722-640_adpp_is (1).mp4"
output_path = "/content/football_tracked.mp4"

# Load YOLO11 model
model = YOLO("yolo11n.pt")

# Track objects throughout the video
results = model.track(
    source=video_path,
    tracker="bytetrack.yaml",
    conf=0.25,
    persist=True,
    save=True,
    project="/content/football_tracking",
    name="output",
    exist_ok=True,
    verbose=False
)

print("Player tracking completed!")
print("Tracked video processing finished.")


In [ ]:

import os

output_folder = "/content/football_tracking/output"

print("Files in output folder:")

for root, dirs, files in os.walk(output_folder):
    for filename in files:
        print(os.path.join(root, filename))


In [ ]:

from IPython.display import Video, display

tracked_video_path = "/content/football_tracking/output/istockphoto-2245307722-640_adpp_is (1).avi"

display(Video(tracked_video_path, embed=True, width=800))


In [ ]:

import os
import glob
import subprocess
from IPython.display import Video, display

# Find the tracked video
videos = glob.glob(
    "/content/football_tracking/output/*"
)

print("Files found:", videos)

avi_files = [f for f in videos if f.lower().endswith(".avi")]

if avi_files:
    input_video = avi_files[0]
    output_video = "/content/football_tracked.mp4"

    # Convert AVI to MP4
    result = subprocess.run([
        "ffmpeg", "-y",
        "-i", input_video,
        "-vcodec", "libx264",
        "-pix_fmt", "yuv420p",
        "-acodec", "aac",
        output_video
    ], capture_output=True, text=True)

    if result.returncode == 0 and os.path.exists(output_video):
        print("Conversion successful!")
        print("MP4 size:", round(os.path.getsize(output_video) / (1024 * 1024), 2), "MB")
        display(Video(output_video, embed=True, width=800))
    else:
        print("Conversion failed:")
        print(result.stderr[-2000:])
else:
    print("No AVI video found. Check the file list above.")


In [ ]:

import cv2
import pandas as pd
from ultralytics import YOLO
from IPython.display import display

video_path = "/content/istockphoto-2245307722-640_adpp_is (1).mp4"
csv_path = "/content/player_tracking_data.csv"

model = YOLO("yolo11n.pt")

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()

records = []

for frame_number, result in enumerate(
    model.track(
        source=video_path,
        tracker="bytetrack.yaml",
        conf=0.25,
        stream=True,
        verbose=False
    ),
    start=1
):
    if result.boxes is None or result.boxes.id is None:
        continue

    boxes = result.boxes.xyxy.cpu().numpy()
    ids = result.boxes.id.int().cpu().tolist()
    confidences = result.boxes.conf.cpu().tolist()

    for box, track_id, confidence in zip(boxes, ids, confidences):
        x1, y1, x2, y2 = box

        records.append({
            "frame": frame_number,
            "time_seconds": (frame_number - 1) / fps if fps else 0,
            "track_id": track_id,
            "x1": float(x1),
            "y1": float(y1),
            "x2": float(x2),
            "y2": float(y2),
            "center_x": float((x1 + x2) / 2),
            "center_y": float((y1 + y2) / 2),
            "confidence": float(confidence)
        })

df = pd.DataFrame(records)
df.to_csv(csv_path, index=False)

print("Tracking data extracted successfully!")
print("Total records:", len(df))
print("CSV saved at:", csv_path)

display(df.head())


In [ ]:

import pandas as pd

# Load the tracking data
df = pd.read_csv('/content/player_tracking_data.csv')

# Display the first 5 records
print("First 5 tracking records:")
display(df.head())

# Count unique player tracking IDs
print("Total unique tracking IDs:", df['track_id'].nunique())

# Count detections for each ID
print("\nTop 10 tracking IDs by detections:")
display(df['track_id'].value_counts().head(10))


In [ ]:

import matplotlib.pyplot as plt

# Count detections for each tracking ID
track_counts = df['track_id'].value_counts().head(10)

# Create the bar graph
plt.figure(figsize=(10, 5))
track_counts.plot(kind='bar')

plt.title('Top 10 Football Tracking IDs by Detection Count')
plt.xlabel('Tracking ID')
plt.ylabel('Number of Detections')
plt.xticks(rotation=45)
plt.tight_layout()

# Display the graph
plt.show()


In [ ]:

import matplotlib.pyplot as plt

# Select one tracking ID
player_id = df['track_id'].value_counts().index[0]

# Filter records for this ID
player_data = df[df['track_id'] == player_id].sort_values('frame')

# Plot movement
plt.figure(figsize=(10, 6))
plt.plot(
    player_data['center_x'],
    player_data['center_y'],
    marker='.',
    markersize=3,
    linestyle='-'
)

plt.title(f'Movement Trajectory for Tracking ID {player_id}')
plt.xlabel('X Position (pixels)')
plt.ylabel('Y Position (pixels)')
plt.gca().invert_yaxis()
plt.grid(True)
plt.tight_layout()
plt.show()

print("Selected tracking ID:", player_id)
print("Number of recorded detections:", len(player_data))


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# Select the tracking ID with the most detections
player_id = df['track_id'].value_counts().index[0]

# Get its tracking records in frame order
player_data = df[df['track_id'] == player_id].sort_values('frame').copy()

# Calculate changes in position and time
dx = player_data['center_x'].diff()
dy = player_data['center_y'].diff()
dt = player_data['time_seconds'].diff()

# Calculate approximate movement speed in pixels per second
player_data['speed_pixels_per_second'] = (
    np.sqrt(dx**2 + dy**2) / dt
).where(dt > 0)

# Remove invalid values for analysis
valid_speeds = player_data['speed_pixels_per_second'].dropna()
valid_speeds = valid_speeds[np.isfinite(valid_speeds)]

print("Tracking ID:", player_id)
print("Average estimated speed:", round(valid_speeds.mean(), 2), "pixels/second")
print("Maximum estimated speed:", round(valid_speeds.max(), 2), "pixels/second")

# Plot estimated speed over time
plt.figure(figsize=(10, 5))
plt.plot(
    player_data['time_seconds'],
    player_data['speed_pixels_per_second'],
    marker='.',
    markersize=3
)
plt.title(f'Estimated Movement Speed - Tracking ID {player_id}')
plt.xlabel('Time (seconds)')
plt.ylabel('Speed (pixels/second)')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:

import numpy as np
import pandas as pd

# Store summary results for each tracking ID
summary = []

for player_id, group in df.groupby('track_id'):
    group = group.sort_values('frame').copy()

    dx = group['center_x'].diff()
    dy = group['center_y'].diff()
    dt = group['time_seconds'].diff()

    speeds = np.sqrt(dx**2 + dy**2) / dt
    speeds = speeds.where(dt > 0).replace([np.inf, -np.inf], np.nan).dropna()

    summary.append({
        'track_id': player_id,
        'detections': len(group),
        'average_speed_pixels_per_second':
            round(speeds.mean(), 2) if not speeds.empty else 0
    })

summary_df = pd.DataFrame(summary)

# Show the top 10 IDs by number of detections
summary_df = summary_df.sort_values(
    'detections', ascending=False
).head(10).reset_index(drop=True)

print("Player Tracking Summary")
display(summary_df)


In [ ]:

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.bar(
    summary_df['track_id'].astype(str),
    summary_df['average_speed_pixels_per_second']
)

plt.title('Average Estimated Speed of Top 10 Tracking IDs')
plt.xlabel('Tracking ID')
plt.ylabel('Average Speed (pixels/second)')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()


In [ ]:

# Save the player tracking summary
summary_file = '/content/player_tracking_summary.csv'

summary_df.to_csv(summary_file, index=False)

print("Player tracking summary saved successfully!")
print("File location:", summary_file)

# Display the saved summary
display(pd.read_csv(summary_file))


In [ ]:

import matplotlib.pyplot as plt

# Select the most frequently detected tracking ID
player_id = df['track_id'].value_counts().index[0]
player_data = df[df['track_id'] == player_id].sort_values('frame')

# Create a dashboard with three graphs
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Graph 1: Detection counts
axes[0].bar(
    summary_df['track_id'].astype(str),
    summary_df['detections']
)
axes[0].set_title('Top 10 Detection Counts')
axes[0].set_xlabel('Tracking ID')
axes[0].set_ylabel('Detections')
axes[0].tick_params(axis='x', rotation=45)

# Graph 2: Average estimated speeds
axes[1].bar(
    summary_df['track_id'].astype(str),
    summary_df['average_speed_pixels_per_second']
)
axes[1].set_title('Average Estimated Speed')
axes[1].set_xlabel('Tracking ID')
axes[1].set_ylabel('Pixels/second')
axes[1].tick_params(axis='x', rotation=45)

# Graph 3: Movement trajectory
axes[2].plot(
    player_data['center_x'],
    player_data['center_y']
)
axes[2].invert_yaxis()
axes[2].set_title(f'Movement Trajectory (ID {player_id})')
axes[2].set_xlabel('X Position (pixels)')
axes[2].set_ylabel('Y Position (pixels)')

plt.tight_layout()

# Save and display dashboard
dashboard_file = '/content/football_analytics_dashboard.png'
plt.savefig(dashboard_file, dpi=150, bbox_inches='tight')
plt.show()

print("Dashboard saved successfully!")
print("File location:", dashboard_file)
